In [2]:
import pyspark.sql, pyspark.sql.functions
from pyspark.sql import SparkSession

# Стартиране на SparkSession 
spark = (
    SparkSession.builder
    .appName("k-means-Example")
    .master("local[*]")
    .getOrCreate()
)
print("SparkSession стартиран успешно")
print("Spark версия:", spark.version)

SparkSession стартиран успешно
Spark версия: 4.0.1


In [3]:
# Зареждане на CSV в DataFrame
df = spark.read.csv('students-online.csv', header=True, inferSchema=True)
# Премахваме ID, ако не участва в модела
df = df.drop("ID")

# Преглед на данните
df.show(5)

+------+---+---------+-------------+-------------+----------------+------------+---------------+---------------+
|Gender|Age|   Degree|PlatformUsage|PlatformHours|OutPlatformHours|Satisfaction|Self-assessment|PreferredFormat|
+------+---+---------+-------------+-------------+----------------+------------+---------------+---------------+
|     Ж| 21|Бакалавър|           Да|            6|               2|           4|              4|          Видео|
|     М| 23| Магистър|           Да|            8|               1|           5|              5|          Видео|
|     Ж| 20|Бакалавър|           Не|            0|               3|           3|              3|          Текст|
|     Ж| 22|Бакалавър|           Да|            5|               1|           4|              4|         Смесен|
|     М| 24| Магистър|           Да|            9|               0|           5|              5|          Видео|
+------+---+---------+-------------+-------------+----------------+------------+---------------+

In [5]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import ClusteringEvaluator

# Подготовка на данните
feature_cols = ["Age","PlatformHours","OutPlatformHours","Satisfaction","Self-assessment"]
# Обединяване на всички характеристики във features колона
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = assembler.transform(df)

# Създаване на модел
kmeans = KMeans(k=2, featuresCol="features")
# Обучение
model = kmeans.fit(data)

# Клъстериране
predictions = model.transform(data)
# Центроиди
centers = model.clusterCenters()

# Показване
predictions.select("features", "prediction").show()
# Показване на центроидите
print("Центроиди:")
for i, center in enumerate(centers):
    print(f"Клъстер {i}: {center}")
    evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="prediction",metricName="silhouette")
score = evaluator.evaluate(predictions)

print("Silhouette score:", score)
wcss = model.summary.trainingCost
print("WCSS:", wcss)

+--------------------+----------+
|            features|prediction|
+--------------------+----------+
|[21.0,6.0,2.0,4.0...|         0|
|[23.0,8.0,1.0,5.0...|         0|
|[20.0,0.0,3.0,3.0...|         1|
|[22.0,5.0,1.0,4.0...|         0|
|[24.0,9.0,0.0,5.0...|         0|
|[21.0,0.0,2.0,2.0...|         1|
|[22.0,7.0,1.0,4.0...|         0|
|[23.0,6.0,0.0,4.0...|         0|
|[20.0,0.0,2.0,2.0...|         1|
|[25.0,10.0,0.0,5....|         0|
|[21.0,4.0,1.0,3.0...|         1|
|[22.0,7.0,0.0,4.0...|         0|
+--------------------+----------+

Центроиди:
Клъстер 0: [22.75   7.25   0.625  4.375  4.375]
Клъстер 1: [20.5   1.    2.    2.5   2.75]
Silhouette score: 0.7544719806662917
WCSS: 55.375
